## Install dependencies

In [ ]:
!pip -q install "openai>=1.40" datasets scikit-learn tqdm python-dotenv huggingface_hub

## Hugging Face login


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste token when prompted

## OpenAI API key & client

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

OpenAI API key: ··········


# OpenAI **o3** Evaluation on FLARE / Open FinLLM Information Extraction tasks

This notebook runs **o3** on the FLARE evaluation datasets used by the *Information-Extraction-Leaderboard*:
- ChanceFocus/flare-ner
- ChanceFocus/flare-finer-ord (may be gated)
- ChanceFocus/flare-finred
- ChanceFocus/flare-causal20-sc
- ChanceFocus/flare-cd
- ChanceFocus/flare-fnxl
- ChanceFocus/flare-fsrl

**Where to run bash commands?**
- If you're in a notebook (Jupyter/Colab/VSCode notebook), run bash by putting `!` in front of the command in a code cell (e.g., `!pip install ...`).
- If you're not in a notebook, run the same commands in your computer Terminal (or VSCode integrated Terminal).

**Before running:**
1) `huggingface_hub.notebook_login()` (HF datasets may require access)
2) Set `OPENAI_API_KEY` for o3
3) Optionally adjust `MAX_SAMPLES` to test quickly before full runs


In [ ]:
# ==== Evaluate OpenAI o3 on FLARE (Information Extraction) tasks ====
# This notebook is designed to be robust to small schema differences across datasets.

import os, json, time, random, re
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

from datasets import load_dataset
from tqdm import tqdm
from openai import OpenAI, APIStatusError, APIError, APIConnectionError, RateLimitError
from sklearn.metrics import accuracy_score, f1_score

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first (e.g., in the cell above)."
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "o3"          # set to "o3" (or "o3-mini" if your org doesn't have o3)
SPLIT = "test"        # most FLARE datasets provide "test"
MAX_SAMPLES = 0       # 0 = full split; set N (e.g., 50/200) for a quick smoke test
QPS_DELAY = 1.05      # throttle between requests
MAX_RETRIES = 6       # backoff retries
CACHE_DIR = Path("o3_ie_cache")
CACHE_DIR.mkdir(exist_ok=True)

# ---- Task registry ----
TASKS = {
    # pair extraction: (entity, type)
    "flare_ner":       {"dataset": "ChanceFocus/flare-ner",        "type": "pairs"},
    "flare_finer_ord": {"dataset": "ChanceFocus/flare-finer-ord",  "type": "triples"},  # may be gated
    # relation extraction: (head, relation, tail)
    "flare_finred":    {"dataset": "ChanceFocus/flare-finred",     "type": "triples"},
    # classification
    "flare_causal20_sc":{"dataset":"ChanceFocus/flare-causal20-sc","type": "choice", "choices": ["causal", "noise"]},
    # sequence labeling (token-level)
    "flare_cd":        {"dataset": "ChanceFocus/flare-cd",         "type": "seq"},
    "flare_fnxl":      {"dataset": "ChanceFocus/flare-fnxl",       "type": "seq"},
    "flare_fsrl":      {"dataset": "ChanceFocus/flare-fsrl",       "type": "seq"},
}

# ---- OpenAI call with retries + optional fallback ----
def get_output_text(resp) -> str:
    # openai>=1.40 provides resp.output_text
    if hasattr(resp, "output_text") and isinstance(resp.output_text, str):
        return resp.output_text
    # fallback: try to join text segments
    try:
        parts = []
        for item in resp.output:
            if item.type == "message":
                for c in item.content:
                    if c.type == "output_text":
                        parts.append(c.text)
        return "\n".join(parts).strip()
    except Exception:
        return str(resp)

def call_o3(prompt: str, model_pref: str = MODEL) -> str:
    def _once(m, use_temp: bool = True):
        # Some reasoning models may validate sampling params strictly.
        # We try temperature=0 for reproducibility, and fall back to default params if rejected.
        if use_temp:
            return client.responses.create(model=m, input=prompt, temperature=0)
        return client.responses.create(model=m, input=prompt)
    m = model_pref
    for k in range(MAX_RETRIES):
        try:
            try:
                resp = _once(m, use_temp=True)
            except APIStatusError as e2:
                if e2.status_code == 400 and 'temperature' in str(e2).lower():
                    resp = _once(m, use_temp=False)
                else:
                    raise
            return get_output_text(resp)
        except APIStatusError as e:
            # If o3 isn't available in this account/org, fallback once to o3-mini
            if e.status_code in (400, 404) and m == "o3":
                m = "o3-mini"
                continue
            # quota errors usually won't recover quickly
            raise
        except (RateLimitError, APIConnectionError, APIError) as e:
            # exponential backoff with jitter
            wait = (2 ** k) + random.random()
            time.sleep(wait)
    raise RuntimeError("Exceeded retries calling the model.")

# ---- Parsing utilities ----
_json_block_re = re.compile(r"(\[[\s\S]*\]|\{[\s\S]*\})")

def try_parse_json(text: str):
    text = (text or "").strip()
    if not text:
        return None
    # direct json
    try:
        return json.loads(text)
    except Exception:
        pass
    # find first json-ish block
    m = _json_block_re.search(text)
    if m:
        s = m.group(1)
        try:
            return json.loads(s)
        except Exception:
            return None
    return None

def norm_str(x: Any) -> str:
    return re.sub(r"\s+", " ", str(x)).strip()

def norm_key(x: Any) -> str:
    return norm_str(x).lower()

def parse_pairs(pred_text: str) -> List[Tuple[str, str]]:
    """
    Parse NER outputs into list of (entity, type).

    Robust to multiple pairs in one line like:
      "SILICON VALLEY BANK, ORG Bank, ORG California, LOC"

    Accepts:
      - JSON: [{"entity": "...", "type": "ORG"}, ...] or [["Apple","ORG"], ...]
      - Text containing repeated "<entity>, <TYPE>" pairs
    """
    j = try_parse_json(pred_text)
    out: List[Tuple[str, str]] = []
    if isinstance(j, list):
        for it in j:
            if isinstance(it, dict):
                ent = it.get("entity") or it.get("text") or it.get("span") or it.get("name")
                typ = it.get("type") or it.get("label") or it.get("tag")
                if ent is not None and typ is not None:
                    out.append((norm_str(ent), norm_str(typ)))
            elif isinstance(it, list) and len(it) >= 2:
                out.append((norm_str(it[0]), norm_str(it[1])))
        return out

    text = (pred_text or "").strip()
    if not text:
        return []

    # Extract common NER types repeatedly
    pair_re = re.compile(r"(.*?),(?:\s*)(ORG|PER|LOC|MISC)\b", re.IGNORECASE | re.DOTALL)
    for m in pair_re.finditer(text):
        ent = norm_str(m.group(1))
        typ = norm_str(m.group(2)).upper()
        if ent and typ and len(ent) <= 200:
            out.append((ent, typ))

    # Fallback: line-by-line "ent, TYPE"
    if not out:
        for line in text.splitlines():
            line = line.strip().strip("-•")
            if not line:
                continue
            if "," in line:
                a, b = line.split(",", 1)
                ent, typ = a.strip(), b.strip()
                if ent and typ:
                    out.append((ent, typ))
    return out

def parse_triples(pred_text: str) -> List[Tuple[str, str, str]]:
    """
    Parse relation-extraction outputs into list of (head, tail, relation).

    Many FLARE relation tasks use the format:
      "head ; tail ; relation"

    Accepts:
      - JSON: [{"head": "...", "tail": "...", "relation": "..."}, ...]
      - Lines: "head ; tail ; relation" OR "head | tail | relation" OR "head, tail, relation"
    """
    j = try_parse_json(pred_text)
    out: List[Tuple[str, str, str]] = []
    if isinstance(j, list):
        for it in j:
            if isinstance(it, dict):
                h = it.get("head") or it.get("h") or it.get("subject")
                t = it.get("tail") or it.get("t") or it.get("object")
                r = it.get("relation") or it.get("rel") or it.get("predicate")
                if h is not None and t is not None and r is not None:
                    out.append((norm_str(h), norm_str(t), norm_str(r)))
            elif isinstance(it, list) and len(it) >= 3:
                out.append((norm_str(it[0]), norm_str(it[1]), norm_str(it[2])))
        return out

    for line in (pred_text or "").splitlines():
        line = line.strip().strip("-•")
        if not line:
            continue

        if line.count(";") >= 2:
            parts = [p.strip() for p in line.split(";")]
            if len(parts) >= 3:
                out.append((parts[0], parts[1], ";".join(parts[2:]).strip()))
            continue

        if "|" in line:
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 3:
                out.append((parts[0], parts[1], "|".join(parts[2:]).strip()))
            continue

        if line.count(",") >= 2:
            parts = [p.strip() for p in line.split(",")]
            out.append((parts[0], parts[1], ",".join(parts[2:]).strip()))
            continue

    return out

_tag_re = re.compile(r"\b(?:B|I)-[A-Za-z0-9_.-]+\b|\bO\b")

def parse_seq_tags(pred_text: str, n: int) -> List[str]:
    """
    Parse sequence labeling outputs into a tag list length n.
    Preferred: JSON list of tags.
    Fallback: extract tags in order from text.
    """
    j = try_parse_json(pred_text)
    if isinstance(j, list) and all(isinstance(x, str) for x in j):
        tags = [norm_str(x) for x in j]
        if len(tags) == n:
            return tags

    # fallback: extract tag tokens from text
    tags = _tag_re.findall(pred_text or "")
    tags = [norm_str(t) for t in tags]
    if len(tags) >= n:
        return tags[:n]
    # last resort: pad with O
    return tags + ["O"] * (n - len(tags))

def parse_choice(pred_text: str, allowed: List[str]) -> str:
    s = norm_key(pred_text)
    for a in allowed:
        if re.search(rf"\b{re.escape(a.lower())}\b", s):
            return a
    # fallback: first token
    tok = s.split()
    return tok[0] if tok else ""

# ---- Gold extraction ----
def get_row_id(row: Dict[str, Any], fallback: int) -> str:
    return str(row.get("id") if row.get("id") is not None else fallback)

def get_prompt(row: Dict[str, Any], dtype: str) -> str:
    """Build a prompt from dataset 'query' plus optional fields like text/tokens."""
    q = (row.get("query") or "").strip()
    t = (row.get("text") or "").strip()

    # Many FLARE rows store the input sentence in `text` and only instructions in `query`,
    # so we always append `text` when present.
    if q and t:
        base = f"{q}\n\nText:\n{t}"
    elif q:
        base = q
    else:
        base = t

    if dtype == "seq":
        toks = get_tokens(row) or []
        if toks:
            # Provide tokens explicitly so the model can align tags.
            base = base + "\n\nTokens:\n" + " ".join([str(x) for x in toks])

    return base.strip()

def get_tokens(row: Dict[str, Any]) -> Optional[List[str]]:
    tok = row.get("token") or row.get("tokens")
    if isinstance(tok, list):
        return [norm_str(x) for x in tok]
    return None

def pick_gold(row: Dict[str, Any], dtype: str) -> Any:
    """
    FLARE schemas differ:
      - For NER/relation tasks, 'answer' is usually the structured target.
        'label' may be token BIO tags, so we prefer 'answer' there.
      - For token-level seq tasks, 'label' is usually the gold tag sequence.
    """
    if dtype in ("pairs", "triples"):
        if row.get("answer") is not None:
            return row["answer"]
        if row.get("label") is not None:
            return row["label"]
        return row.get("gold")
    if dtype == "seq":
        if row.get("label") is not None:
            return row["label"]
        if row.get("answer") is not None:
            return row["answer"]
        return row.get("gold")

    if row.get("answer") is not None:
        return row["answer"]
    if row.get("gold") is not None:
        return row["gold"]
    return row.get("label")


def gold_pairs(g) -> List[Tuple[str, str]]:
    # label/answer may already be list[dict] or list[list] or string lines
    if isinstance(g, list):
        out = []
        for it in g:
            if isinstance(it, dict):
                ent = it.get("entity") or it.get("text") or it.get("span") or it.get("name")
                typ = it.get("type") or it.get("label") or it.get("tag")
                if ent is not None and typ is not None:
                    out.append((norm_str(ent), norm_str(typ)))
            elif isinstance(it, list) and len(it) >= 2:
                out.append((norm_str(it[0]), norm_str(it[1])))
            elif isinstance(it, str):
                # try "ent, TYPE"
                if "," in it:
                    a,b = it.split(",",1)
                    out.append((a.strip(), b.strip()))
        return out
    if isinstance(g, str):
        return parse_pairs(g)
    return []

def gold_triples(g) -> List[Tuple[str, str, str]]:
    if isinstance(g, list):
        out=[]
        for it in g:
            if isinstance(it, dict):
                h = it.get("head") or it.get("h") or it.get("subject")
                t = it.get("tail") or it.get("t") or it.get("object")
                r = it.get("relation") or it.get("rel") or it.get("predicate")
                if h is not None and t is not None and r is not None:
                    out.append((norm_str(h), norm_str(t), norm_str(r)))
            elif isinstance(it, list) and len(it) >= 3:
                out.append((norm_str(it[0]), norm_str(it[1]), norm_str(it[2])))
            elif isinstance(it, str):
                # parse line
                out += parse_triples(it)
        return out
    if isinstance(g, str):
        return parse_triples(g)
    return []

def gold_seq(g, n: int) -> List[str]:
    if isinstance(g, list) and all(isinstance(x, str) for x in g):
        tags = [norm_str(x) for x in g]
        if len(tags) == n:
            return tags
    # try parse from string
    if isinstance(g, str):
        return parse_seq_tags(g, n)
    # fallback all O
    return ["O"] * n

# ---- Metrics ----
def micro_prf(correct: int, pred_n: int, gold_n: int) -> Dict[str, float]:
    p = correct / pred_n if pred_n else 0.0
    r = correct / gold_n if gold_n else 0.0
    f1 = (2*p*r/(p+r)) if (p+r) else 0.0
    return {"precision": p, "recall": r, "f1": f1}

def score_set_pairs(pred: List[Tuple[str,str]], gold: List[Tuple[str,str]]) -> Dict[str,float]:
    ps = {(norm_key(a), norm_key(b)) for a,b in pred if a and b}
    gs = {(norm_key(a), norm_key(b)) for a,b in gold if a and b}
    correct = len(ps & gs)
    return micro_prf(correct, len(ps), len(gs))

def score_set_triples(pred: List[Tuple[str,str,str]], gold: List[Tuple[str,str,str]]) -> Dict[str,float]:
    ps = {(norm_key(a), norm_key(b), norm_key(c)) for a,b,c in pred if a and b and c}
    gs = {(norm_key(a), norm_key(b), norm_key(c)) for a,b,c in gold if a and b and c}
    correct = len(ps & gs)
    return micro_prf(correct, len(ps), len(gs))

def score_seq(pred_tags: List[str], gold_tags: List[str]) -> Dict[str,float]:
    assert len(pred_tags) == len(gold_tags)
    # token-level micro F1 ignoring "O" as negative class
    correct = sum((p==g) and (g!="O") for p,g in zip(pred_tags, gold_tags))
    pred_pos = sum(p!="O" for p in pred_tags)
    gold_pos = sum(g!="O" for g in gold_tags)
    m = micro_prf(correct, pred_pos, gold_pos)
    em = sum(p==g for p,g in zip(pred_tags, gold_tags)) / len(gold_tags) if gold_tags else 0.0
    m["token_accuracy"] = em
    return m

# ---- Main runner ----
def resolve_split(ds_dict, preferred: str):
    if preferred in ds_dict:
        return preferred
    for s in ["test", "validation", "valid", "dev", "train"]:
        if s in ds_dict:
            return s
    return list(ds_dict.keys())[0]

def load_split(dataset_name: str, split_pref: str):
    ds = load_dataset(dataset_name)
    sp = resolve_split(ds, split_pref) if isinstance(ds, dict) else split_pref
    if isinstance(ds, dict):
        return ds[sp], sp
    return ds, split_pref

def eval_task(task_key: str) -> Dict[str, Any]:
    cfg = TASKS[task_key]
    dsname = cfg["dataset"]
    dtype  = cfg["type"]
    cache_path = CACHE_DIR / f"{task_key}__{dsname.replace('/','__')}__{SPLIT}.jsonl"

    print(f"\n=== {task_key}  ({dsname}) ===")
    data, used_split = load_split(dsname, SPLIT)
    print(f"Loaded split: {used_split}  rows={len(data)}")

    # load cache
    cache = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["id"]] = obj
                except Exception:
                    pass
        print(f"Cache: {len(cache)} rows")

    # loop
    ids, preds_parsed, golds_parsed = [], [], []
    gold_choices, pred_choices = [], []

    n_total = len(data) if MAX_SAMPLES==0 else min(MAX_SAMPLES, len(data))
    for i in tqdm(range(n_total)):
        row = data[i]
        rid = get_row_id(row, i)
        prompt = get_prompt(row, dtype)

        # optional strictness to make parsing easier while still using the dataset query
        if dtype == "pairs":
            prompt2 = prompt + "\n\nReturn ONLY the final answer. Prefer JSON list of {entity,type}."
        elif dtype == "triples":
            prompt2 = prompt + "\n\nReturn ONLY the final answer. Prefer JSON list of {head,relation,tail}."
        
elif dtype == "seq":
    tok = get_tokens(row) or []
    # Encourage a clean, parseable output.
    prompt2 = (
        prompt
        + f"\n\nReturn ONLY a JSON list of tags (strings) with length={len(tok)}."
        + "\nDo not include any explanation or extra text."
    )
        elif dtype == "choice":
            prompt2 = prompt + "\n\nReturn ONLY one word: causal OR noise."
        else:
            prompt2 = prompt

        if rid in cache:
            pred_text = cache[rid]["pred_text"]
        else:
            pred_text = call_o3(prompt2, MODEL)
            with cache_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"id": rid, "pred_text": pred_text}, ensure_ascii=False) + "\n")
            time.sleep(QPS_DELAY)

        g = pick_gold(row, dtype)

        if dtype == "pairs":
            pred = parse_pairs(pred_text)
            gold = gold_pairs(g)
            preds_parsed.append(pred)
            golds_parsed.append(gold)
        elif dtype == "triples":
            pred = parse_triples(pred_text)
            gold = gold_triples(g)
            preds_parsed.append(pred)
            golds_parsed.append(gold)
        elif dtype == "seq":
            tok = get_tokens(row) or []
            n = len(tok)
            pred = parse_seq_tags(pred_text, n)
            gold = gold_seq(g, n)
            preds_parsed.append(pred)
            golds_parsed.append(gold)
        elif dtype == "choice":
            allowed = cfg.get("choices", ["causal","noise"])
            pred = parse_choice(pred_text, allowed)
            gold = parse_choice(str(g), allowed)
            pred_choices.append(pred)
            gold_choices.append(gold)
        else:
            raise ValueError("Unknown dtype")

        ids.append(rid)

    # compute metrics
    if dtype == "pairs":
        agg_correct = agg_pred = agg_gold = 0
        for p,g in zip(preds_parsed, golds_parsed):
            ps = {(norm_key(a), norm_key(b)) for a,b in p if a and b}
            gs = {(norm_key(a), norm_key(b)) for a,b in g if a and b}
            agg_correct += len(ps & gs)
            agg_pred += len(ps)
            agg_gold += len(gs)
        m = micro_prf(agg_correct, agg_pred, agg_gold)

    elif dtype == "triples":
        agg_correct = agg_pred = agg_gold = 0
        for p,g in zip(preds_parsed, golds_parsed):
            ps = {(norm_key(a), norm_key(b), norm_key(c)) for a,b,c in p if a and b and c}
            gs = {(norm_key(a), norm_key(b), norm_key(c)) for a,b,c in g if a and b and c}
            agg_correct += len(ps & gs)
            agg_pred += len(ps)
            agg_gold += len(gs)
        m = micro_prf(agg_correct, agg_pred, agg_gold)

    elif dtype == "seq":
        # aggregate token-level (ignore O)
        agg_correct = agg_pred = agg_gold = 0
        acc_sum = 0.0
        tok_count = 0
        for p,g in zip(preds_parsed, golds_parsed):
            correct = sum((pp==gg) and (gg!="O") for pp,gg in zip(p,g))
            pred_pos = sum(pp!="O" for pp in p)
            gold_pos = sum(gg!="O" for gg in g)
            agg_correct += correct
            agg_pred += pred_pos
            agg_gold += gold_pos
            acc_sum += sum(pp==gg for pp,gg in zip(p,g))
            tok_count += len(g)
        m = micro_prf(agg_correct, agg_pred, agg_gold)
        m["token_accuracy"] = acc_sum / tok_count if tok_count else 0.0

    elif dtype == "choice":
        acc = accuracy_score(gold_choices, pred_choices) if gold_choices else 0.0
        f1  = f1_score(gold_choices, pred_choices, average="macro") if gold_choices else 0.0
        m = {"accuracy": float(acc), "macro_f1": float(f1)}
    else:
        m = {}

    # save task-level results
    out = {
        "task": task_key,
        "dataset": dsname,
        "split": used_split,
        "model": MODEL,
        "n": int(n_total),
        "metrics": m,
    }
    print("Metrics:", json.dumps(m, indent=2))
    return out

# ---- Run all tasks ----
results = []
for task_key in TASKS.keys():
    try:
        results.append(eval_task(task_key))
    except Exception as e:
        print(f"[WARN] Failed task {task_key}: {e}")

# ---- Save a compact results JSON ----
out_path = Path("o3_openfinllm_ie_results.json")
out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved: {out_path.resolve()}")


Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']


Querying o3 on test (N=970): 100%|██████████| 970/970 [00:00<00:00, 11104.37it/s]


=== Summary ===
Model: o3  Dataset: TheFinAI/en-fpb [test]  N=970
Accuracy: 0.8113 | F1(micro): 0.8113 | F1(macro): 0.8144

Per-class:
    positive  P=0.681 R=0.884 F1=0.769 (support=277)
     neutral  P=0.941 R=0.740 F1=0.828 (support=577)
    negative  P=0.737 R=0.991 F1=0.846 (support=116)

Saved:
- o3_eval_outputs/predictions__o3__TheFinAI__en-fpb__test.tsv
- o3_eval_outputs/metrics__o3__TheFinAI__en-fpb__test.json
- cache: o3_eval_cache__TheFinAI__en-fpb__test.jsonl
